# COLMAP Dense Reconstruction (GPU)

This notebook runs COLMAP with **GPU-accelerated dense reconstruction** on Google Colab.
It produces much higher quality 3D models than CPU-only sparse reconstruction.

**Requirements:** Use a GPU runtime (Runtime → Change runtime type → T4 GPU)

## How it works
1. Downloads coral reef images from HuggingFace
2. Runs COLMAP sparse reconstruction (feature extraction, matching, mapping)
3. Runs COLMAP **dense reconstruction** (undistortion, patch_match_stereo, stereo_fusion)
4. Converts dense point cloud to GLB mesh using Open3D
5. Downloads the model

In [ ]:
# Install COLMAP with CUDA support
# The apt version does NOT have CUDA, so we build from source

import subprocess, os

# Check GPU
!nvidia-smi | head -4
print()

# Install build dependencies
print("Installing build dependencies...")
!apt-get update -qq > /dev/null 2>&1
!apt-get install -y -qq \
    cmake ninja-build build-essential git \
    libboost-program-options-dev libboost-filesystem-dev \
    libboost-graph-dev libboost-system-dev \
    libeigen3-dev libflann-dev libfreeimage-dev \
    libmetis-dev libgoogle-glog-dev libgflags-dev \
    libsqlite3-dev libceres-dev libglew-dev \
    qtbase5-dev libqt5opengl5-dev libcgal-dev \
    > /dev/null 2>&1
print("Build dependencies installed ✓")

# Clone COLMAP
if not os.path.exists("/tmp/colmap"):
    !git clone --branch 3.9.1 --depth 1 https://github.com/colmap/colmap.git /tmp/colmap
    print("COLMAP source cloned ✓")
else:
    print("COLMAP source already cloned ✓")

# Build with CUDA
os.makedirs("/tmp/colmap/build", exist_ok=True)

print("\nRunning cmake...")
result = subprocess.run(
    ["cmake", "..", "-GNinja",
     "-DCMAKE_BUILD_TYPE=Release",
     "-DCMAKE_INSTALL_PREFIX=/usr/local",
     "-DCUDA_ENABLED=ON",
     "-DGUI_ENABLED=OFF",
     "-DCGAL_ENABLED=ON"],
    cwd="/tmp/colmap/build",
    capture_output=True, text=True
)
if result.returncode != 0:
    print("CMAKE FAILED:")
    print(result.stderr[-2000:])
    raise RuntimeError("cmake failed")
print("cmake configured ✓")

print("\nBuilding COLMAP (this takes ~5-10 min on Colab)...")
result = subprocess.run(
    ["ninja", "-j4"],
    cwd="/tmp/colmap/build",
    capture_output=True, text=True,
    timeout=1200
)
if result.returncode != 0:
    print("BUILD FAILED:")
    print(result.stderr[-2000:])
    raise RuntimeError("build failed")
print("Build complete ✓")

print("\nInstalling...")
!cd /tmp/colmap/build && ninja install > /dev/null 2>&1
!ldconfig

# Verify
!which colmap
!colmap -h 2>&1 | head -3

print("\n✓ COLMAP with CUDA installed successfully!")

# Install Python deps
!pip install -q open3d trimesh numpy

In [ ]:
# Configuration
NUM_IMAGES = 30  # Number of coral images to download
START_INDEX = 7546
BASE_URL = "https://huggingface.co/datasets/wildflow/sweet-corals/resolve/main/indonesia_pemuteran_p1_20250213/raw/B1_Left"

In [ ]:
import os
import subprocess
from pathlib import Path

# Set up directories
WORKDIR = Path("/content/colmap_project")
IMAGE_DIR = WORKDIR / "images"
SPARSE_DIR = WORKDIR / "sparse"
DENSE_DIR = WORKDIR / "dense"
DB_PATH = WORKDIR / "database.db"

for d in [IMAGE_DIR, SPARSE_DIR, DENSE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Download images
print(f"Downloading {NUM_IMAGES} coral reef images...")
for i in range(START_INDEX, START_INDEX + NUM_IMAGES):
    filename = f"GPAA{i}.JPG"
    filepath = IMAGE_DIR / filename
    if not filepath.exists():
        !wget -q -O {filepath} "{BASE_URL}/{filename}"
        print(f"  {filename} ({filepath.stat().st_size / 1024 / 1024:.1f} MB)")
    else:
        print(f"  {filename} (cached)")

print(f"\nDownloaded {NUM_IMAGES} images")

In [ ]:
def run_colmap(args, stage_name):
    """Run a COLMAP command and print progress."""
    cmd = ["colmap"] + args
    print(f"\n{'='*60}")
    print(f"Stage: {stage_name}")
    print(f"Command: {' '.join(cmd[:4])}...")
    print(f"{'='*60}")
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=1800)
    if result.returncode != 0:
        print(f"FAILED!\nstderr: {result.stderr[-1000:]}")
        raise RuntimeError(f"{stage_name} failed")
    print(f"SUCCESS")
    return result

In [ ]:
# Stage 1: Feature Extraction (GPU-accelerated)
run_colmap([
    "feature_extractor",
    "--database_path", str(DB_PATH),
    "--image_path", str(IMAGE_DIR),
    "--ImageReader.single_camera", "1",
    "--ImageReader.camera_model", "SIMPLE_RADIAL",
    "--SiftExtraction.use_gpu", "1",
    "--SiftExtraction.max_num_features", "8192",
], "Feature Extraction")

In [ ]:
# Stage 2: Feature Matching (GPU-accelerated)
run_colmap([
    "exhaustive_matcher",
    "--database_path", str(DB_PATH),
    "--SiftMatching.use_gpu", "1",
], "Feature Matching")

In [ ]:
# Stage 3: Sparse Reconstruction (Mapper)
run_colmap([
    "mapper",
    "--database_path", str(DB_PATH),
    "--image_path", str(IMAGE_DIR),
    "--output_path", str(SPARSE_DIR),
], "Sparse Reconstruction")

# Find best model
sparse_models = sorted(SPARSE_DIR.iterdir())
print(f"\nSparse models found: {[m.name for m in sparse_models]}")
SPARSE_MODEL = sparse_models[0]
print(f"Using model: {SPARSE_MODEL}")

In [ ]:
# Stage 4: Image Undistortion (prepare for dense reconstruction)
run_colmap([
    "image_undistorter",
    "--image_path", str(IMAGE_DIR),
    "--input_path", str(SPARSE_MODEL),
    "--output_path", str(DENSE_DIR),
    "--output_type", "COLMAP",
], "Image Undistortion")

In [ ]:
# Stage 5: Dense Stereo (GPU - this is the key step!)
# This computes depth maps for every image using GPU patch matching
run_colmap([
    "patch_match_stereo",
    "--workspace_path", str(DENSE_DIR),
    "--workspace_format", "COLMAP",
    "--PatchMatchStereo.geom_consistency", "true",
], "Dense Stereo (GPU)")

In [ ]:
# Stage 6: Stereo Fusion (merge depth maps into dense point cloud)
DENSE_PLY = WORKDIR / "dense_reconstruction.ply"
run_colmap([
    "stereo_fusion",
    "--workspace_path", str(DENSE_DIR),
    "--workspace_format", "COLMAP",
    "--output_path", str(DENSE_PLY),
], "Stereo Fusion")

print(f"\nDense PLY size: {DENSE_PLY.stat().st_size / 1024 / 1024:.1f} MB")

In [ ]:
# Stage 7: Convert to GLB mesh
import numpy as np
import open3d as o3d
import trimesh

print("Loading dense point cloud...")
pcd = o3d.io.read_point_cloud(str(DENSE_PLY))
print(f"Loaded {len(pcd.points)} points")

# Clean outliers
pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
print(f"After outlier removal: {len(pcd.points)} points")

# Estimate normals
bbox = pcd.get_axis_aligned_bounding_box()
extent = np.linalg.norm(bbox.get_max_bound() - bbox.get_min_bound())
radius = extent / 30.0
pcd.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=radius, max_nn=50)
)
pcd.orient_normals_consistent_tangent_plane(k=30)

# Poisson reconstruction at high detail
print("Running Poisson surface reconstruction (depth=11)...")
mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=11)

# Trim low-density noise
densities = np.asarray(densities)
threshold = np.percentile(densities, 2)
mesh.remove_vertices_by_mask(densities < threshold)

print(f"Mesh: {len(mesh.vertices)} vertices, {len(mesh.triangles)} triangles")

# Export GLB
vertices = np.asarray(mesh.vertices)
faces = np.asarray(mesh.triangles)
vertex_colors = None
if mesh.has_vertex_colors():
    vertex_colors = (np.asarray(mesh.vertex_colors) * 255).astype(np.uint8)

tri_mesh = trimesh.Trimesh(vertices=vertices, faces=faces, vertex_colors=vertex_colors)

OUTPUT_GLB = WORKDIR / "coral_model_dense.glb"
tri_mesh.export(str(OUTPUT_GLB), file_type="glb")
print(f"\nExported to {OUTPUT_GLB} ({OUTPUT_GLB.stat().st_size / 1024 / 1024:.1f} MB)")

In [ ]:
# Also export the sparse reconstruction for comparison
SPARSE_PLY = WORKDIR / "sparse_reconstruction.ply"
run_colmap([
    "model_converter",
    "--input_path", str(SPARSE_MODEL),
    "--output_path", str(SPARSE_PLY),
    "--output_type", "PLY",
], "Export Sparse PLY")
print(f"Sparse PLY: {SPARSE_PLY.stat().st_size / 1024:.0f} KB")
print(f"Dense PLY:  {DENSE_PLY.stat().st_size / 1024 / 1024:.1f} MB")
print(f"\nDense has ~{DENSE_PLY.stat().st_size // SPARSE_PLY.stat().st_size}x more data!")

In [ ]:
# Download the model
from google.colab import files
files.download(str(OUTPUT_GLB))
print("\nDone! Open the .glb file in https://gltf-viewer.donmccurdy.com to view it.")